# The Lab — Local vs Cloud LLM Benchmark

Companion notebook for `docs/BENCHMARK_LOCAL_VS_CLOUD.md`.
Compares the multi-agent pipeline driven by a **local Ollama model** (llama3.2:3b) vs **cloud OpenRouter** (GLM 5.3 Flash) on the Titanic dataset.

## 1. Ingest the dataset from Kaggle

In [ ]:
from thelab.ide.kaggle_api import ingest_kaggle_dataset, fetch_kaggle_page_context, build_context_pack

SLUG = "yasserh/titanic-dataset"
ingestion = ingest_kaggle_dataset(SLUG)
page = fetch_kaggle_page_context(SLUG)
pack = build_context_pack(SLUG, ingestion, page)
print("dataset:", ingestion["dataset_id"])
print("description:", (pack.get("description_markdown") or "")[:200])

## 2. Clean (deterministic policy)

In [ ]:
from thelab.ide.cleaning import clean_dataset

cleaned = clean_dataset(ingestion["dataset_id"], target="Survived")
print(cleaned["dataset_id"], f"{cleaned['rows']} rows x {cleaned['columns']} cols")
for action in cleaned["cleaning_report"]["actions"]:
    print("-", action)

## 3. Multi-agent experiment — local Ollama

In [ ]:
import asyncio
from thelab.ide.experiment_api import start_experiment

started = asyncio.run(start_experiment(
    goal="Predict Titanic survival",
    dataset_id=cleaned["dataset_id"],
    target="Survived",
    provider_name="ollama",
))
print("experiment:", started["experiment_id"], "| provider: ollama (llama3.2:3b)")
# Wait for completion (or poll via /experiment/{id}/status)

## 4. Same experiment — cloud OpenRouter (GLM 5.3 Flash)

In [ ]:
started_cloud = asyncio.run(start_experiment(
    goal="Predict Titanic survival",
    dataset_id=cleaned["dataset_id"],
    target="Survived",
    provider_name="openrouter",
))
print("experiment:", started_cloud["experiment_id"], "| provider: openrouter")

## 4b. Deterministic-only path (no LLM)

Run the same training without any agent — the ground-truth baseline that validates the design.

In [ ]:
from thelab.run.runner import run_model

result_det = run_model(
    dataset=cleaned["dataset_id"],
    target="Survived",
    model="logistic_regression",
    seed=42,
    output="scratch",
    task_type="classification",
)
m = result_det["metrics"]
print(f"deterministic: acc={m['test_accuracy']:.4f} f1={m['test_f1_macro']:.4f}")
print("(identical to Ollama and OpenRouter paths — verified in docs/BENCHMARK_LOCAL_VS_CLOUD.md)")

## 5. Compare interpretations

In [ ]:
import json
from thelab.ide.experiment_api import get_experiment_status

for label, exp_id in [("Ollama", started["experiment_id"]), ("OpenRouter", started_cloud["experiment_id"])]:
    status = asyncio.run(get_experiment_status(exp_id))
    print(f"\n=== {label} ===")
    print("state:", status["state"])
    for role, block in (status.get("sub_agent_results") or {}).items():
        interp = block.get("llm_interpretation")
        usage = block.get("llm_usage")
        if interp:
            print(f"  {role}: {str(interp)[:200]}")
        if usage:
            print(f"  tokens: {json.dumps(usage, default=str)}")
    if status.get("best_metrics"):
        print("  best metrics:", {k: round(v, 4) for k, v in status["best_metrics"].items() if isinstance(v, float)})

## 6. The key takeaway

The deterministic pipeline produces **identical training results** regardless of LLM provider. The LLM adds **interpretation and reasoning** — the quality of which varies with model size and capability, but never changes the training outcome.

This is the thesis design: **LLM decides → deterministic pipeline executes → every result is reproducible**.